# Part 3: Final Model

<div dir="rtl">

این نوت‌بوک سومین و آخرین نوت‌بوک بخش سوم است. مدل TF-IDF (از part3_sentiment) و چهار ترنسفورمر (از part3_finetune) احتمال هر امتیاز را برای validation، holdout و test ذخیره کرده‌اند. اینجا هیچ مدلی آموزش نمی‌بیند؛ فقط همین احتمال‌ها به کار می‌روند و اجرا روی CPU چند دقیقه طول می‌کشد.

مدل‌هایی که برای انتخاب مدل نهایی مقایسه شدند TF-IDF، MiniLM، RoBERTa و DeBERTa هستند. DistilRoBERTa بعد از انتخاب مدل نهایی اضافه شد تا مدلی که یکی از اعضای گروه جداگانه آموزش داده بود با همان داده و همان ارزیابی سنجیده شود. این مدل در همه‌ی جدول‌ها گزارش می‌شود، ولی در ترکیب و در انتخاب مدل نهایی نیست تا آن تصمیم‌ها بعد از دیدن holdout تغییر نکنند.

مراحل:

1. مقایسه‌ی چهار مدل روی validation.
2. ترکیب مدل‌ها با میانگین وزنی احتمال‌ها.
3. انتخاب مدل نهایی با قاعده‌ای که از قبل تعیین شده بود.
4. یک بار ارزیابی روی holdout.
5. تحلیل خطا: ماتریس درهم‌ریختگی، طول نظر و مطمئن‌ترین خطاها.
6. درستی روی داده‌ای با ترکیب امتیازهای train، برای مقایسه با عددهایی که روی آن نوع داده گزارش می‌شوند.
7. پیش‌بینی test و ساختن q2_submission.csv.
8. ذخیره‌ی جدول‌های داشبورد.

فایل‌های لازم در پوشه‌ی project2 در Google Drive: part3_train.parquet و part3_test.parquet، part3_tfidf_probs.parquet و part3_tfidf.joblib، و پوشه‌ی finetune (probs.parquet و history.csv هر مدل). خروجی‌ها: q2_submission.csv در همان پوشه و فایل‌های داشبورد در outputs/part3.

</div>

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import itertools
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

pio.templates.default = 'plotly_white'
pd.set_option('display.max_colwidth', 300)
path = '/content/drive/MyDrive/project2/'

## Data and saved predictions

<div dir="rtl">

برچسب‌ها و متن‌های validation و holdout از part3_train.parquet خوانده شدند. احتمال‌های هر مدل با index همان سطر ذخیره شده‌اند. پس قبل از هر مقایسه بررسی شد که هر چهار مدل دقیقاً همان نظرها را دارند، و احتمال‌ها به یک ترتیب مشترک درآمدند.

</div>

In [3]:
reviews = pd.read_parquet(path + 'part3_train.parquet', columns=['overall', 'summary', 'reviewText', 'set'],
                          filters=[('set', 'in', ['valid', 'holdout'])])
test = pd.read_parquet(path + 'part3_test.parquet')
index = {'valid': reviews.index[reviews['set'] == 'valid'], 'holdout': reviews.index[reviews['set'] == 'holdout'],
         'test': test.index}
y = {s: reviews.loc[index[s], 'overall'].values for s in ['valid', 'holdout']}

files = {'TF-IDF': 'part3_tfidf_probs.parquet', 'MiniLM': 'finetune/minilm/probs.parquet',
         'DistilRoBERTa': 'finetune/distilroberta/probs.parquet', 'RoBERTa': 'finetune/roberta/probs.parquet',
         'DeBERTa': 'finetune/deberta/probs.parquet'}
cols = ['p1', 'p2', 'p3', 'p4', 'p5']
probs = {}
for name, file in files.items():
    p = pd.read_parquet(path + file)
    assert all(set(p.index[p['set'] == s]) == set(index[s]) for s in index), name
    probs[name] = {s: p.loc[p['set'] == s, cols].loc[index[s]].values for s in index}

print({s: len(i) for s, i in index.items()}, ' models:', list(probs))
print('ratings in validation:', np.bincount(y['valid'])[1:], ' in holdout:', np.bincount(y['holdout'])[1:])

{'valid': 10000, 'holdout': 10000, 'test': 20000}  models: ['TF-IDF', 'MiniLM', 'DistilRoBERTa', 'RoBERTa', 'DeBERTa']
ratings in validation: [2000 2000 2000 2000 2000]  in holdout: [2000 2000 2000 2000 2000]


## Comparison on validation

<div dir="rtl">

چهار معیار روی validation حساب شد:

- **accuracy:** سهم نظرهایی که امتیازشان درست پیش‌بینی شده. معیار پروژه micro-F1 است که در دسته‌بندی تک‌برچسبی دقیقاً برابر accuracy است.
- **macro-F1:** میانگین F1 پنج امتیاز. روی داده‌ی متوازن تقریباً با accuracy برابر است.
- **log-loss:** میانگین منفی لگاریتم احتمالی که مدل به امتیاز درست داده. هرچه کمتر، احتمال‌ها بهتر. برای ترکیب مدل‌ها مهم است، چون ترکیب روی احتمال‌ها انجام می‌شود.
- **within one star:** سهم پیش‌بینی‌هایی که حداکثر یک ستاره با امتیاز واقعی فاصله دارند.

</div>

In [4]:
def predicted(p):
    return p.argmax(1) + 1

def log_loss(p, y_true):
    return -np.log(np.clip(p[np.arange(len(y_true)), y_true - 1], 1e-12, 1)).mean()

def metrics(p, y_true):
    pred = predicted(p)
    return {'accuracy': accuracy_score(y_true, pred), 'macro_f1': f1_score(y_true, pred, average='macro'),
            'log_loss': log_loss(p, y_true), 'within_one_star': (np.abs(pred - y_true) <= 1).mean()}

valid_table = pd.DataFrame({name: metrics(p['valid'], y['valid']) for name, p in probs.items()}).T
valid_table.round(4)

,accuracy,macro_f1,log_loss,within_one_star
TF-IDF,0.6032,0.6001,0.9566,0.9217
MiniLM,0.6723,0.6725,0.7807,0.9666
DistilRoBERTa,0.6789,0.6786,0.7536,0.9695
RoBERTa,0.6920,0.6920,0.7268,0.9768
DeBERTa,0.7033,0.7030,0.7165,0.9790


<div dir="rtl">

recall هر امتیاز، یعنی سهم نظرهای آن امتیاز که درست پیش‌بینی شده‌اند:

</div>

In [5]:
recall = pd.DataFrame({name: [(predicted(p['valid'])[y['valid'] == r] == r).mean() for r in range(1, 6)]
                       for name, p in probs.items()}, index=[f'rating {r}' for r in range(1, 6)]).T
recall.round(3)

,rating 1,rating 2,rating 3,rating 4,rating 5
TF-IDF,0.696,0.464,0.511,0.559,0.786
MiniLM,0.727,0.588,0.591,0.648,0.808
DistilRoBERTa,0.730,0.600,0.597,0.640,0.828
RoBERTa,0.745,0.600,0.616,0.664,0.836
DeBERTa,0.759,0.610,0.622,0.676,0.849


<div dir="rtl">

ترتیب مدل‌ها با هر چهار معیار یکی است: DeBERTa (۰.۷۰۳۳)، RoBERTa (۰.۶۹۲۰)، DistilRoBERTa (۰.۶۷۸۹)، MiniLM (۰.۶۷۲۳) و TF-IDF (۰.۶۰۳۲). log-loss مدل DeBERTa (۰.۷۱۶۵) خیلی کمتر از TF-IDF (۰.۹۵۶۶) است، یعنی احتمال‌هایش هم مطمئن‌تر و هم درست‌ترند. ۹۷.۹٪ پیش‌بینی‌های DeBERTa حداکثر یک ستاره خطا دارند، در برابر ۹۲.۲٪ برای TF-IDF.

بیشترین سود ترنسفورمرها در امتیازهای میانی است. recall امتیاز ۲ از ۰.۴۶۴ در TF-IDF به ۰.۶۱۰ در DeBERTa رسید (۱۴.۶ امتیاز)، و امتیاز ۳ و ۴ هر کدام حدود ۱۱ تا ۱۲ امتیاز بهتر شدند. امتیاز ۱ و ۵ فقط ۶.۳ امتیاز بهتر شدند. نظرهای ۱ و ۵ معمولاً کلمه‌های صریحی مثل junk یا perfect دارند که TF-IDF هم آن‌ها را می‌گیرد. نظرهای میانی ترکیبی از تعریف و انتقادند و وزن هر بخش به زمینه‌ی جمله بستگی دارد. امتیازهای ۲ و ۳ برای هر پنج مدل سخت‌ترین امتیازها هستند.

</div>

## Ensemble

<div dir="rtl">

ترکیب ساده‌ای امتحان شد: میانگین وزنی احتمال‌های چهار مدلی که برای انتخاب مقایسه می‌شوند (TF-IDF، MiniLM، RoBERTa و DeBERTa). وزن‌ها نامنفی‌اند و جمعشان ۱ است. همه‌ی ترکیب‌های وزن با گام ۰.۰۵ امتحان شدند (۱۷۷۱ ترکیب). اگر وزن یک مدل صفر شود، یعنی آن مدل چیزی به ترکیب اضافه نمی‌کند.

وزن‌ها با log-loss روی validation انتخاب شدند، نه با accuracy. accuracy فقط به امتیاز برنده‌ی هر نظر بستگی دارد و با تغییر کوچک وزن‌ها پله‌ای بالا و پایین می‌رود. پس وزنی که بیشترین accuracy را بدهد، بیشتر نویز همین ۱۰ هزار نظر را دنبال می‌کند. log-loss پیوسته است و به همه‌ی احتمال‌ها حساس است.

</div>

In [6]:
names = ['TF-IDF', 'MiniLM', 'RoBERTa', 'DeBERTa']  # selection candidates; DistilRoBERTa is reported only
stack = {s: np.stack([probs[n][s] for n in names]) for s in index}
grid = np.array([w for w in itertools.product(range(21), repeat=len(names)) if sum(w) == 20]) / 20

def best_weights(rows):
    losses = [log_loss(np.tensordot(w, stack['valid'][:, rows], axes=1), y['valid'][rows]) for w in grid]
    return grid[int(np.argmin(losses))]

weights = best_weights(np.arange(len(y['valid'])))
ensemble = {s: np.tensordot(weights, stack[s], axes=1) for s in index}
print('combinations tried:', len(grid))
print('weights:', dict(zip(names, weights.round(2).tolist())))
print('ensemble on validation:', {k: round(float(v), 4) for k, v in metrics(ensemble['valid'], y['valid']).items()})

combinations tried: 1771
weights: {'TF-IDF': 0.05, 'MiniLM': 0.1, 'RoBERTa': 0.3, 'DeBERTa': 0.55}
ensemble on validation: {'accuracy': 0.7051, 'macro_f1': 0.7048, 'log_loss': 0.6932, 'within_one_star': 0.9795}


<div dir="rtl">

وزن‌ها روی همان validation انتخاب شده‌اند که ترکیب با آن سنجیده می‌شود، پس accuracy بالا کمی خوش‌بینانه است. برای مقایسه‌ی منصفانه با مدل‌های تکی، validation به ۵ بخش تصادفی تقسیم شد. هر بار وزن‌ها روی ۴ بخش انتخاب شدند و ترکیب روی بخش پنجم پیش‌بینی کرد (out-of-fold)، پس هیچ نظری با وزن‌هایی که از خودش یاد گرفته شده پیش‌بینی نشد.

</div>

In [7]:
fold = np.random.default_rng(0).permutation(len(y['valid'])) % 5
oof_pred = np.zeros(len(y['valid']), dtype=int)
for k in range(5):
    w = best_weights(np.where(fold != k)[0])
    oof_pred[fold == k] = predicted(np.tensordot(w, stack['valid'][:, fold == k], axes=1))
    print(f'fold {k}: weights', dict(zip(names, w.round(2).tolist())))
print('out-of-fold ensemble accuracy on validation:', round(accuracy_score(y['valid'], oof_pred), 4))

fold 0: weights {'TF-IDF': 0.05, 'MiniLM': 0.15, 'RoBERTa': 0.25, 'DeBERTa': 0.55}
fold 1: weights {'TF-IDF': 0.05, 'MiniLM': 0.1, 'RoBERTa': 0.3, 'DeBERTa': 0.55}
fold 2: weights {'TF-IDF': 0.05, 'MiniLM': 0.1, 'RoBERTa': 0.3, 'DeBERTa': 0.55}
fold 3: weights {'TF-IDF': 0.05, 'MiniLM': 0.1, 'RoBERTa': 0.3, 'DeBERTa': 0.55}
fold 4: weights {'TF-IDF': 0.05, 'MiniLM': 0.15, 'RoBERTa': 0.25, 'DeBERTa': 0.55}
out-of-fold ensemble accuracy on validation: 0.7055


<div dir="rtl">

وزن‌ها روی کل validation: DeBERTa ۰.۵۵، RoBERTa ۰.۳، MiniLM ۰.۱ و TF-IDF ۰.۰۵. هیچ وزنی صفر نشد، یعنی هر مدل کمی اطلاعات تازه به ترکیب اضافه می‌کند. log-loss ترکیب ۰.۶۹۳۲ است در برابر ۰.۷۱۶۵ برای DeBERTa تنها، ولی درستی‌اش فقط کمی بیشتر است (۰.۷۰۵۱ در برابر ۰.۷۰۳۳). وزن‌های پنج fold تقریباً یکی‌اند: DeBERTa همیشه ۰.۵۵ است و وزن بقیه حداکثر ۰.۰۵ جابه‌جا می‌شود. پس وزن‌ها پایدارند و درستی out-of-fold (۰.۷۰۵۵) با درستی روی همان داده (۰.۷۰۵۱) فرقی ندارد.

</div>

## Choosing the final model

<div dir="rtl">

قاعده‌ی انتخاب پیش از دیدن این نتیجه‌ها تعیین شده بود: مدلی که روی validation بهترین است، مگر اینکه مدل سبک‌تری با فاصله‌ای در حد نویز پشتش باشد؛ در آن صورت مدل سبک‌تر انتخاب می‌شود. سبک‌تر یعنی محاسبه‌ی کمتر برای هر پیش‌بینی. ترتیب از سبک به سنگین: TF-IDF، MiniLM، RoBERTa، DeBERTa و در آخر ترکیب، که هر چهار مدل را لازم دارد. برای ترکیب، درستی out-of-fold به کار رفت.

نویز با paired bootstrap سنجیده شد. ۲۰۰۰ بار، ۱۰ هزار نظر از validation با جایگذاری انتخاب شد و هر بار فاصله‌ی درستی هر مدل با بهترین مدل روی همان نمونه حساب شد. اگر بازه‌ی ۹۵٪ این فاصله‌ها صفر را در بر بگیرد، فاصله در حد نویز است. paired یعنی هر دو مدل همیشه روی همان نظرها مقایسه می‌شوند. این بازه خیلی تنگ‌تر از مقایسه‌ی دو عدد جداست، چون نظری که برای یک مدل سخت است معمولاً برای مدل دیگر هم سخت است.

</div>

In [8]:
correct = pd.DataFrame({name: predicted(probs[name]['valid']) == y['valid'] for name in names})
correct['Ensemble'] = oof_pred == y['valid']
best = correct.mean().idxmax()

rng = np.random.default_rng(0)
gaps = []
for _ in range(2000):
    sample = correct.values[rng.integers(0, len(correct), len(correct))].mean(axis=0)
    gaps.append(sample - sample[correct.columns.get_loc(best)])
gaps = np.array(gaps)

selection = pd.DataFrame({'accuracy': correct.mean(), 'gap_to_best': correct.mean() - correct[best].mean(),
                          'ci_low': np.percentile(gaps, 2.5, axis=0), 'ci_high': np.percentile(gaps, 97.5, axis=0)})
selection['within_noise'] = (selection['ci_low'] <= 0) & (selection['ci_high'] >= 0)
final = selection.index[selection['within_noise']][0]
print('best on validation:', best, ' final model:', final)
selection.round(4)

best on validation: Ensemble  final model: DeBERTa


,accuracy,gap_to_best,ci_low,ci_high,within_noise
TF-IDF,0.6032,-0.1023,-0.1120,-0.0926,False
MiniLM,0.6723,-0.0332,-0.0403,-0.0263,False
RoBERTa,0.6920,-0.0135,-0.0191,-0.0083,False
DeBERTa,0.7033,-0.0022,-0.0060,0.0015,True
Ensemble,0.7055,0.0000,0.0000,0.0000,True


In [9]:
fig = px.scatter(selection.reset_index(names='model'), x='gap_to_best', y='model', error_x_minus=selection['gap_to_best'] - selection['ci_low'],
                 error_x=selection['ci_high'] - selection['gap_to_best'],
                 title=f'Accuracy gap to the best model ({best}) on validation, with 95% bootstrap interval')
fig.add_vline(x=0, line_dash='dot')
fig.update_layout(xaxis_tickformat='.1%', height=350)
fig.show()

<div dir="rtl">

بهترین مدل روی validation ترکیب است (۰.۷۰۵۵). DeBERTa تنها ۰.۲ امتیاز عقب‌تر است و بازه‌ی ۹۵٪ این فاصله از ۰.۶ امتیاز کمتر تا ۰.۱۵ امتیاز بیشتر است. این بازه صفر را در بر می‌گیرد، پس فاصله در حد نویز است. RoBERTa ۱.۳۵ امتیاز عقب است و کل بازه‌اش (۰.۸ تا ۱.۹ امتیاز کمتر) زیر صفر است. MiniLM و TF-IDF هم خیلی عقب‌ترند.

پس طبق قاعده DeBERTa-v3-base مدل نهایی است. درستی‌اش در حد نویز همان درستی ترکیب است، و برای هر پیش‌بینی به‌جای چهار مدل فقط یک مدل لازم دارد.

</div>

## Holdout: one-time evaluation

<div dir="rtl">

holdout تا اینجا در هیچ تصمیمی به کار نرفته است: نه در آموزش، نه در انتخاب بهترین checkpoint، نه در وزن‌های ترکیب و نه در انتخاب مدل نهایی. انتخاب بالا ثابت است و عددهای این بخش فقط گزارش‌اند. بقیه‌ی مدل‌ها هم برای مقایسه آمده‌اند؛ ترکیب با وزن‌هایی که روی کل validation انتخاب شد.

</div>

In [10]:
all_probs = probs | {'Ensemble': ensemble}
final_probs = all_probs[final]
holdout_table = pd.DataFrame({name: metrics(p['holdout'], y['holdout']) for name, p in all_probs.items()}).T
holdout_table.insert(0, 'valid_accuracy', valid_table['accuracy'].reindex(holdout_table.index))
holdout_table.loc['Ensemble', 'valid_accuracy'] = correct['Ensemble'].mean()

final_correct = predicted(final_probs['holdout']) == y['holdout']
boot = [final_correct[rng.integers(0, len(final_correct), len(final_correct))].mean() for _ in range(2000)]
print(f'{final} on holdout: accuracy {final_correct.mean():.4f}, 95% interval {np.percentile(boot, 2.5):.4f} to {np.percentile(boot, 97.5):.4f}')
holdout_table.round(4)

DeBERTa on holdout: accuracy 0.6949, 95% interval 0.6859 to 0.7041


,valid_accuracy,accuracy,macro_f1,log_loss,within_one_star
TF-IDF,0.6032,0.5941,0.5910,0.9649,0.9229
MiniLM,0.6723,0.6646,0.6653,0.7813,0.9674
DistilRoBERTa,0.6789,0.6825,0.6826,0.7493,0.9706
RoBERTa,0.6920,0.6927,0.6926,0.7291,0.9767
DeBERTa,0.7033,0.6949,0.6945,0.7258,0.9785
Ensemble,0.7055,0.6967,0.6965,0.7006,0.9793


<div dir="rtl">

DeBERTa روی holdout ۰.۶۹۴۹ درست است، با بازه‌ی ۹۵٪ از ۰.۶۸۶ تا ۰.۷۰۴. این عدد ۰.۸ امتیاز کمتر از عددش روی validation (۰.۷۰۳۳) است. چنین افتی انتظار می‌رفت: بهترین checkpoint هر مدل با همان validation انتخاب شده بود، پس عدد validation کمی خوش‌بینانه است. macro-F1 (۰.۶۹۴۵) تقریباً همان درستی است و ۹۷.۸۵٪ پیش‌بینی‌ها حداکثر یک ستاره خطا دارند.

ترتیب مدل‌ها روی holdout همان ترتیب validation است. DistilRoBERTa، که فقط گزارش می‌شود، روی holdout ۰.۶۸۲۵ است: ۱ امتیاز کمتر از RoBERTa، با نصف لایه‌ها. ترکیب روی holdout ۰.۱۸ امتیاز بهتر از DeBERTa است (۰.۶۹۶۷)، تقریباً همان فاصله‌ای که validation نشان داد. یعنی انتخاب DeBERTa به‌جای ترکیب روی holdout حدود ۰.۲ امتیاز هزینه داشت، در حد همان نویزی که قاعده در نظر گرفته بود.

</div>

## Error analysis

<div dir="rtl">

همه‌ی تحلیل‌های این بخش روی holdout و برای مدل نهایی است.

</div>

In [11]:
holdout_pred = predicted(final_probs['holdout'])
cm = confusion_matrix(y['holdout'], holdout_pred, normalize='true')
labels = ['1', '2', '3', '4', '5']
fig = px.imshow(cm, x=labels, y=labels, text_auto='.2f', color_continuous_scale='Blues',
                labels=dict(x='predicted rating', y='true rating', color='share'),
                title=f'{final} on holdout (each row sums to 1)')
fig.show()
pd.DataFrame(classification_report(y['holdout'], holdout_pred, output_dict=True)).T.iloc[:5].round(3)

,precision,recall,f1-score,support
1,0.748,0.754,0.751,2000.0
2,0.572,0.586,0.579,2000.0
3,0.640,0.620,0.629,2000.0
4,0.697,0.675,0.686,2000.0
5,0.815,0.840,0.828,2000.0


<div dir="rtl">

تقریباً همه‌ی خطاها به امتیاز کناری می‌روند. برای هیچ امتیازی، خطای دو ستاره یا بیشتر از ۴٪ نظرها بیشتر نیست. امتیاز ۵ (F1 برابر ۰.۸۲۸) و ۱ (۰.۷۵۱) آسان‌ترین‌اند و امتیاز ۲ (۰.۵۷۹) سخت‌ترین: ۲۲٪ نظرهای ۲ ستاره، ۱ و ۱۸٪ آن‌ها ۳ پیش‌بینی شده‌اند. خطاهای امتیاز ۳ بیشتر به سمت ۲ می‌روند (۲۱٪) تا ۴ (۱۳٪)، یعنی مدل نظرهای ۳ ستاره را بیشتر منفی می‌خواند تا مثبت. خطاهای امتیاز ۴ بیشتر به سمت ۵ است (۱۸٪).

</div>

### Review length

<div dir="rtl">

طول هر نظر به token با tokenizer هر مدل شمرده شد، به همان شکلی که به مدل داده شده بود: summary تا ۲۰۰ کاراکتر و متن نظر، به‌صورت یک جفت. جدول اول نشان می‌دهد چه سهمی از نظرهای validation در ۲۵۶ و ۵۱۲ token جا می‌شوند. بقیه‌ی نظرها موقع آموزش و پیش‌بینی کوتاه شده‌اند. جدول دوم درستی مدل نهایی و مدل TF-IDF را در گروه‌های طول روی holdout مقایسه می‌کند. TF-IDF هیچ محدودیت طولی ندارد و همه‌ی متن را می‌بیند.

</div>

In [12]:
import transformers
from transformers import AutoTokenizer

transformers.logging.set_verbosity_error()
tokenizer_names = {'MiniLM': 'sentence-transformers/all-MiniLM-L6-v2', 'RoBERTa': 'FacebookAI/roberta-base',
                   'DeBERTa': 'microsoft/deberta-v3-base'}

def token_lengths(tokenizer, df):
    ids = tokenizer(df['summary'].str[:200].tolist(), df['reviewText'].tolist())['input_ids']
    return np.array([len(x) for x in ids])

valid_reviews = reviews.loc[index['valid']]
coverage = {}
for name, model_name in tokenizer_names.items():
    lengths = token_lengths(AutoTokenizer.from_pretrained(model_name), valid_reviews)
    coverage[name] = {'within_256': (lengths <= 256).mean(), 'within_512': (lengths <= 512).mean(),
                      'median_tokens': np.median(lengths), 'p90_tokens': np.percentile(lengths, 90)}
coverage = pd.DataFrame(coverage).T
coverage.round(3)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

,within_256,within_512,median_tokens,p90_tokens
MiniLM,0.871,0.969,108.0,296.0
RoBERTa,0.877,0.971,106.0,286.0
DeBERTa,0.878,0.972,105.0,285.0


In [13]:
holdout_lengths = token_lengths(AutoTokenizer.from_pretrained(tokenizer_names['DeBERTa']), reviews.loc[index['holdout']])
groups = pd.cut(holdout_lengths, [0, 64, 128, 256, 512, np.inf], labels=['up to 64', '65-128', '129-256', '257-512', 'over 512'])
by_length = pd.DataFrame({'group': groups, 'final': final_correct,
                          'TF-IDF': predicted(probs['TF-IDF']['holdout']) == y['holdout']})
by_length = by_length.groupby('group', observed=True).agg(reviews=('final', 'size'), final=('final', 'mean'), tfidf=('TF-IDF', 'mean'))
by_length = by_length.rename(columns={'final': final + ' accuracy', 'tfidf': 'TF-IDF accuracy'})
fig = px.bar(by_length.drop(columns='reviews').reset_index().melt(id_vars='group', var_name='model', value_name='accuracy'),
             x='group', y='accuracy', color='model', barmode='group', text_auto='.2f',
             title='Holdout accuracy by review length (DeBERTa tokens)')
fig.show()
by_length.round(3)

,reviews,DeBERTa accuracy,TF-IDF accuracy
group,,,
up to 64,1196,0.690,0.627
65-128,4864,0.693,0.598
129-256,2730,0.694,0.579
257-512,954,0.724,0.584
over 512,256,0.664,0.562


<div dir="rtl">

در validation، ۸۷ تا ۸۸ درصد نظرها در ۲۵۶ token جا می‌شوند و ۹۷ درصد در ۵۱۲ token. سه tokenizer تقریباً یکسان‌اند و میانه‌ی طول ۱۰۵ تا ۱۰۸ token است. پس با ۵۱۲ token فقط حدود ۳٪ نظرها کوتاه شدند.

درستی DeBERTa در گروه‌های طول تقریباً ثابت است (۰.۶۹۰ تا ۰.۷۲۴). نظرهای بالای ۵۱۲ token، که انتهایشان دیده نشده، ۰.۶۶۴ گرفتند. ولی این گروه فقط ۲۵۶ نظر دارد و با این تعداد، چند امتیاز فرق را نمی‌شود از نویز جدا کرد.

TF-IDF برعکس، با بلندتر شدن نظر ضعیف‌تر می‌شود: ۰.۶۲۷ برای نظرهای تا ۶۴ token و حدود ۰.۵۸ برای نظرهای ۱۲۹ تا ۵۱۲ token. سود DeBERTa نسبت به TF-IDF در نظرهای کوتاه ۶.۳ امتیاز است و در نظرهای ۲۵۷ تا ۵۱۲ token به ۱۴ امتیاز می‌رسد. یک توضیح محتمل: نظر بلند معمولاً هم تعریف دارد و هم انتقاد. مدل کلمه‌محور اثر همه‌ی کلمه‌ها را با هم جمع می‌زند، ولی ترنسفورمر می‌تواند ببیند هر جمله درباره‌ی چیست و نویسنده در آخر چه نتیجه‌ای گرفته.

</div>

### Most confident mistakes

<div dir="rtl">

نظرهایی که مدل نهایی با بیشترین اطمینان اشتباه پیش‌بینی کرده و پیش‌بینی‌اش دست‌کم سه ستاره با امتیاز واقعی فاصله دارد:

</div>

In [14]:
holdout_view = reviews.loc[index['holdout'], ['overall', 'summary', 'reviewText']].assign(
    predicted=holdout_pred, confidence=final_probs['holdout'].max(1))
far = holdout_view[(holdout_view['predicted'] - holdout_view['overall']).abs() >= 3]
print('mistakes of three or more stars:', len(far), f'({len(far) / len(holdout_view):.1%} of holdout)')
far.sort_values('confidence', ascending=False).head(8).assign(reviewText=lambda d: d['reviewText'].str[:300]).round(3)

mistakes of three or more stars: 27 (0.3% of holdout)


,overall,summary,reviewText,predicted,confidence
777385,4,"Badly packed, broken to heck","Updated Apr 25 2016 Once I got one that arrived intact, handy little tool box. I would buy again, long as they pack it right Busted to heck inside and it was used because the screwdriver already had a bit in it. I have ordered another kit, but the 1 star review remains until the new kit arrives ...",1,0.994
794624,4,DOES NOT WORK WITH EL CAPITAN -,"I am crushed... Opened the box, tried to pair with my iMac, nothing. Plugged it in to charge up, thinking it was a matter of power, the light turned green, still nothing. The computer could not 'see' the keyboard. I must have been doing something wrong, right? So I re-read the manual (nice instr...",1,0.832
621447,1,Purchase a charger it may not arrive with the tablet,"I Just receive the tablet it works very well but i had to put 1 start since the seller dint send the charger , this tablet use a very small jack Dc 5v and i don t have one to charge the tablet , it does not charge with USB it needs the dc in",4,0.752
523622,5,"65"" Sony 4K TV","So far so good.. I have read all the reviews prior to buying, so I went into this knowing some people had issues. The picture is very clear, I don't use the TV speakers due to I have a Visio Surround System for it. But overall I am pleased.. Will update later if anything changes. UPDATE:: 8/22/1...",1,0.749
673674,1,Nice Sounding BT speaker,These were good. Bought two to use one upstairs and one down. One is already being returned as it is constantly losing bluetooth connection and shutting off. Sound and connection to multiple bluetooth devices is very easy. Cool blue LED lights under handle when power is on. About the size of a t...,4,0.704
69815,1,Great Antenna,"So far so good. I have bought a few indoor antennas and this seems to work the best. I like that it has its own stand. You do however after find the best position. I live in a condo in Los Angeles with buildings around me. The signal strength seems good I can get channels 2, 4, 5 clearly. I coul...",5,0.689
734445,2,More expensive isn't always better,"After months of back and forth with Canon about fax operating properly, they suggested I try another cable, and son of a gun- it was the cable the whole time! I had a cheapie lying around it it worked fine! No more fax issues",5,0.629
518431,4,any tone 10 meter ham radio /transciver,any tone smart mobile radio/ transceiver attention people this is not a c.b radio this is a ham radio the transmits on ten meter freqs you must have a ham radio operators lincese tech class or general class license other wise you are violating F.C.C rules and regs if caught with this radio you c...,1,0.616


<div dir="rtl">

فقط ۲۷ نظر از ۱۰ هزار نظر holdout (۰.۳٪) سه ستاره یا بیشتر خطا دارند. در بیشتر نمونه‌های بالا متن با امتیاز جور نیست، و کسی که فقط متن را بخواند هم احتمالاً همان پیش‌بینی مدل را می‌کرد:

- **نظرهای به‌روزشده:** نظر اول متن شکایت قبلی را دارد و حتی می‌نویسد امتیاز ۱ ستاره می‌ماند تا کیت جدید برسد، ولی امتیازش ۴ است. برعکسش هم هست: نظری که با «تا اینجا خوب» شروع شده و بعد در UPDATE خاموش شدن تلویزیون را گزارش کرده، با امتیاز ۵ مانده.
- **امتیاز کم برای چیزی غیر از خود کالا:** «تبلت خیلی خوب کار می‌کند، ولی چون فروشنده شارژر نفرستاد ۱ ستاره دادم.»
- **مشکلی که بعداً حل شده:** نظر ۲ ستاره‌ای که آخرش می‌گوید مشکل از کابل بود و حالا همه‌چیز درست کار می‌کند.

این نوع خطا با مدل بهتر رفع نمی‌شود، چون اطلاعات لازم در متن نیست. سقف درستی هر مدلی که فقط متن را می‌بیند را همین نظرها پایین می‌آورند.

</div>

## Accuracy on data with the training mix

<div dir="rtl">

test این پروژه متوازن است، ولی در train امتیاز ۵ بیش از نیمی از نظرهاست. درستی یک مدل به ترکیب امتیازهای داده‌ی ارزیابی بستگی دارد. پس عددی که روی داده‌ای با ترکیب train به دست می‌آید، مثلاً وقتی validation به‌طور تصادفی از train جدا شود، با عدد روی داده‌ی متوازن قابل مقایسه نیست.

درستی روی داده‌ای با ترکیب train از recall هر امتیاز روی holdout حساب شد: جمع سهم هر امتیاز در train ضرب در recall همان امتیاز. فرض این محاسبه این است که نظرهای هر امتیاز همان نظرهای holdout باشند و فقط تعدادشان عوض شود. دو حالت حساب شد:

- **as trained:** مدل همان‌طور که روی داده‌ی متوازن آموزش دیده.
- **shifted to train mix:** احتمال هر امتیاز در سهم آن امتیاز در train ضرب و بر ۰.۲ تقسیم شد. این همان تغییری است که مدلی که روی داده‌ی طبیعی آموزش ببیند خودبه‌خود یاد می‌گیرد: وقتی ۵ ستاره خیلی رایج است، در موارد مردد ۵ گفتن بیشتر درست درمی‌آید.

</div>

In [15]:
labels_all = pd.read_parquet(path + 'part3_train.parquet', columns=['overall', 'set'])
prior = labels_all.loc[labels_all['set'] == 'train', 'overall'].value_counts(normalize=True).sort_index().values
print('share of each rating in train:', prior.round(3))

def per_rating_recall(pred):
    return np.array([(pred[y['holdout'] == r] == r).mean() for r in range(1, 6)])

rows = {}
for name, p in all_probs.items():
    trained = per_rating_recall(predicted(p['holdout']))
    shifted = per_rating_recall(predicted(p['holdout'] * prior / 0.2))
    rows[name] = {'balanced, as trained': trained.mean(), 'train mix, as trained': prior @ trained,
                  'balanced, shifted': shifted.mean(), 'train mix, shifted': prior @ shifted}
train_mix = pd.DataFrame(rows).T
train_mix.round(4)

share of each rating in train: [0.096 0.064 0.094 0.187 0.558]


,"balanced, as trained","train mix, as trained","balanced, shifted","train mix, shifted"
TF-IDF,0.5941,0.6700,0.5251,0.7122
MiniLM,0.6646,0.7221,0.6482,0.7613
DistilRoBERTa,0.6825,0.7445,0.6593,0.7745
RoBERTa,0.6927,0.7583,0.6753,0.7866
DeBERTa,0.6949,0.7640,0.6850,0.7919
Ensemble,0.6967,0.7641,0.6824,0.7935


<div dir="rtl">

روی holdout متوازن، DeBERTa ۰.۶۹۴۹ درست است. اگر همین نظرها با ترکیب train آمده بودند، همین مدل بدون هیچ تغییری ۰.۷۶۴۰ درست بود، و با انتقال احتمال‌ها به ترکیب train ۰.۷۹۱۹. یعنی حدود ۱۰ امتیاز فرق فقط از ترکیب امتیازهای داده‌ی ارزیابی می‌آید، نه از بهتر شدن مدل. همان انتقال، درستی روی داده‌ی متوازن را از ۰.۶۹۴۹ به ۰.۶۸۵۰ پایین می‌آورد.

پس درستی حدود ۰.۷۹ روی داده‌ای با ترکیب train (RoBERTa، DeBERTa و ترکیب) با حدود ۰.۶۹ تا ۰.۷۰ روی داده‌ی متوازن هم‌ارز است. برای test متوازن این پروژه، مدلی درست است که برای سهم برابر امتیازها تنظیم شده باشد، و آن همین مدل‌های آموزش‌دیده روی داده‌ی متوازن‌اند.

</div>

## Test predictions and submission

<div dir="rtl">

test برچسب ندارد، ولی سهم امتیازهای پیش‌بینی‌شده روی test را می‌شود با دو حالت مقایسه کرد: سهم پیش‌بینی‌ها روی holdout، که واقعاً متوازن است، و سهمی که همین مدل پیش‌بینی می‌کرد اگر test ترکیب train را داشت. حالت دوم از ماتریس درهم‌ریختگی holdout و سهم امتیازها در train حساب شد.

</div>

In [16]:
test_pred = predicted(final_probs['test'])
shares = pd.DataFrame({'test, predicted': np.bincount(test_pred, minlength=6)[1:] / len(test_pred),
                       'holdout (balanced), predicted': np.bincount(holdout_pred, minlength=6)[1:] / len(holdout_pred),
                       'if test had the train mix': prior @ cm}, index=[1, 2, 3, 4, 5])
fig = px.bar(shares.reset_index(names='rating').melt(id_vars='rating', var_name='data', value_name='share'),
             x='rating', y='share', color='data', barmode='group', text_auto='.0%',
             title=f'Share of each predicted rating ({final})')
fig.update_layout(xaxis_type='category', yaxis_tickformat='.0%')
fig.show()
shares.round(3)

,"test, predicted","holdout (balanced), predicted",if test had the train mix
1,0.202,0.202,0.091
2,0.205,0.205,0.081
3,0.193,0.194,0.101
4,0.195,0.194,0.222
5,0.203,0.206,0.504


<div dir="rtl">

سهم امتیازهای پیش‌بینی‌شده روی test بین ۱۹.۳٪ و ۲۰.۵٪ است و تقریباً دقیقاً همان سهم‌های holdout است (حداکثر ۰.۳ امتیاز فرق). اگر test ترکیب train را داشت، همین مدل ۵۰٪ نظرها را ۵ پیش‌بینی می‌کرد، نه ۲۰٪. این با برآورد EM و BBSE در نوت‌بوک اول (۱۶٪ تا ۲۳٪ برای هر امتیاز) جور است و تا اینجا قوی‌ترین نشانه است که test متوازن است.

</div>

<div dir="rtl">

فایل submission: یک ستون predicted، بدون index، با همان ترتیب test_data.csv.

</div>

In [17]:
submission = pd.DataFrame({'predicted': test_pred})
assert len(submission) == len(test) == 20000
assert (test.index == np.arange(len(test))).all()
assert submission['predicted'].isin([1, 2, 3, 4, 5]).all()
submission.to_csv(path + 'q2_submission.csv', index=False)

saved = pd.read_csv(path + 'q2_submission.csv')
print('rows:', len(saved), ' columns:', list(saved.columns))
print(saved.head())

rows: 20000  columns: ['predicted']
   predicted
0          1
1          1
2          1
3          1
4          1


## Row order of the test file

<div dir="rtl">

بعد از ذخیره‌ی فایل submission، یک بررسی دیگر روی پیش‌بینی‌های test انجام شد: سهم هر امتیاز پیش‌بینی‌شده در بلوک‌های ۲۰۰۰ سطری، به همان ترتیب سطرهای test_data.csv. این بررسی فقط از پیش‌بینی‌های مدل استفاده می‌کند.

</div>

In [18]:
block_shares = pd.crosstab(np.arange(len(test)) // 2000, test_pred, normalize='index')
block_labels = [f'rows {b * 2000}-{b * 2000 + 1999}' for b in block_shares.index]
fig = px.imshow(block_shares.values, x=[str(c) for c in block_shares.columns], y=block_labels, text_auto='.2f',
                color_continuous_scale='Blues', labels=dict(x='predicted rating', y='test rows', color='share'),
                title=f'Predicted ratings ({final}) in blocks of 2,000 test rows', height=500, aspect='auto')
fig.show()

<div dir="rtl">

پیش‌بینی‌ها بلوک‌بلوک‌اند. در ۴۰۰۰ سطر اول حدود ۷۵٪ پیش‌بینی‌ها ۱ است، در ۴۰۰۰ سطر بعد حدود ۶۰٪ آن‌ها ۲، و همین‌طور تا ۴۰۰۰ سطر آخر که حدود ۸۵٪ پیش‌بینی‌ها ۵ است. هر جفت بلوک تقریباً همان یک سطر از ماتریس درهم‌ریختگی holdout است؛ مثلاً ۴۰۰۰ سطر اول ۰.۷۵ و ۰.۲۱ و ۰.۰۳ دارند، مثل نظرهای ۱ ستاره‌ی holdout. پس سطرهای test_data.csv به ترتیب امتیاز مرتب شده‌اند و از هر امتیاز حدود ۴۰۰۰ نظر دارند. این همان متوازن بودن test را تأیید می‌کند.

ترتیب سطرها در هیچ مرحله‌ای استفاده نشد. مدل فقط متن را می‌بیند، فایل submission پیش از این بررسی ذخیره شد و هیچ تصمیمی بعد از آن عوض نشد. پیش‌بینی از روی شماره‌ی سطر تحلیل احساس نیست؛ فقط از نحوه‌ی ساخته شدن فایل استفاده می‌کند.

با فرض اینکه ترتیب دقیقاً بر اساس امتیاز باشد (سطرهای ۰ تا ۳۹۹۹ امتیاز ۱، ۴۰۰۰ تا ۷۹۹۹ امتیاز ۲ و همین‌طور تا ۵)، می‌شود درستی مدل‌ها روی test را تخمین زد. این تخمین فقط برای سنجیدن روش ارزیابی است و در انتخاب مدل اثری ندارد:

</div>

In [19]:
order_rating = np.arange(len(test)) // 4000 + 1
order_check = pd.DataFrame({'holdout_accuracy': holdout_table['accuracy'],
                            'test_accuracy_if_sorted': {name: (predicted(p['test']) == order_rating).mean()
                                                        for name, p in all_probs.items()}})
order_check.round(4)

,holdout_accuracy,test_accuracy_if_sorted
TF-IDF,0.5941,0.6040
MiniLM,0.6646,0.6702
DistilRoBERTa,0.6825,0.6841
RoBERTa,0.6927,0.6953
DeBERTa,0.6949,0.6989
Ensemble,0.6967,0.7046


<div dir="rtl">

درستی تخمینی DeBERTa روی test ۰.۶۹۸۹ است. این عدد داخل بازه‌ی ۹۵٪ holdout (۰.۶۸۶ تا ۰.۷۰۴) و نزدیک ۰.۶۹۴۹ است. پس holdout تخمین درستی از عملکرد روی test داد، و ترتیب مدل‌های تکی روی test همان ترتیب validation و holdout است.

ترکیب روی test ۰.۶ امتیاز از DeBERTa بهتر است (۰.۷۰۴۶)، بیشتر از ۰.۲ امتیازی که validation و holdout نشان دادند. مدل نهایی با این عدد عوض نمی‌شود. انتخاب مدل بر اساس test همان نشتی است که validation و holdout برای جلوگیری از آن جدا شدند، و قاعده‌ی انتخاب پیش از این بررسی اجرا شده بود.

</div>

## Files for the dashboard

<div dir="rtl">

جدول‌ها و پیش‌بینی‌هایی که داشبورد نمایش می‌دهد در outputs/part3 ذخیره می‌شوند. مدل TF-IDF برای پیش‌بینی زنده به‌صورت آرایه‌های ساده ذخیره می‌شود (idf، وزن‌ها، عرض از مبدأ و فهرست کلمات) تا داشبورد به نسخه‌ی scikit-learn وابسته نباشد. همه‌ی این فایل‌ها در یک zip هم جمع می‌شوند.

</div>

In [20]:
import json
import joblib
import zipfile

out = path + 'outputs/part3/'
os.makedirs(out, exist_ok=True)

minutes = {}
history = []
for run in ['minilm_small_256', 'minilm_small_256_headtail', 'minilm_small_512', 'minilm_small_256_lr1e-4', 'minilm', 'distilroberta',
            'roberta', 'deberta']:
    h = pd.read_csv(path + 'finetune/' + run + '/history.csv').assign(run=run)
    history.append(h)
    minutes[run] = h['minutes'].max()
pd.concat(history).to_csv(out + 'transformer_history.csv', index=False)

comparison = holdout_table.rename(columns={c: 'holdout_' + c for c in holdout_table.columns if c != 'valid_accuracy'})
comparison['million_parameters'] = pd.Series({'MiniLM': 22.7, 'DistilRoBERTa': 82, 'RoBERTa': 125, 'DeBERTa': 184})
comparison['training_minutes'] = pd.Series({'MiniLM': minutes['minilm'], 'DistilRoBERTa': minutes['distilroberta'],
                                            'RoBERTa': minutes['roberta'], 'DeBERTa': minutes['deberta']})
comparison['final'] = comparison.index == final
comparison['selection_candidate'] = comparison.index.isin(names + ['Ensemble'])
comparison.rename_axis('model').to_csv(out + 'model_comparison.csv')
selection.rename_axis('model').to_csv(out + 'model_selection.csv')
pd.Series(weights, index=names, name='weight').rename_axis('model').to_csv(out + 'ensemble_weights.csv')
recall.rename_axis('model').to_csv(out + 'valid_recall.csv')
coverage.rename_axis('tokenizer').to_csv(out + 'length_coverage.csv')
by_length.to_csv(out + 'accuracy_by_length.csv')
train_mix.rename_axis('model').to_csv(out + 'train_mix_accuracy.csv')
shares.rename_axis('rating').to_csv(out + 'test_prediction_shares.csv')
order_check.rename_axis('model').to_csv(out + 'test_order_check.csv')
pd.DataFrame(block_shares.values, index=block_labels, columns=[f'predicted {c}' for c in block_shares.columns]).rename_axis(
    'rows').to_csv(out + 'test_row_blocks.csv')
holdout_all = pd.DataFrame({'overall': y['holdout']}, index=index['holdout'])
for name, p in all_probs.items():
    holdout_all[name + ' predicted'] = predicted(p['holdout'])
    holdout_all[name + ' confidence'] = p['holdout'].max(1)
holdout_all.to_parquet(out + 'holdout_predictions.parquet')

holdout_view.assign(**{f'p{r}': final_probs['holdout'][:, r - 1] for r in range(1, 6)}).to_parquet(out + 'final_holdout_predictions.parquet')
valid_reviews[['overall', 'summary', 'reviewText']].assign(
    predicted=predicted(probs['TF-IDF']['valid']), confidence=probs['TF-IDF']['valid'].max(1)).to_parquet(out + 'tfidf_valid_predictions.parquet')

bundle = joblib.load(path + 'part3_tfidf.joblib')
vectorizer, model = bundle['vectorizer'], bundle['model']
terms = np.empty(len(vectorizer.vocabulary_), dtype=object)
for term, i in vectorizer.vocabulary_.items():
    terms[i] = term
np.savez_compressed(out + 'tfidf_model.npz', idf=vectorizer.idf_.astype(np.float32), coef=model.coef_.astype(np.float32),
                    intercept=model.intercept_.astype(np.float32), classes=model.classes_)
json.dump(list(terms), open(out + 'tfidf_vocabulary.json', 'w'))

with zipfile.ZipFile(path + 'part3_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(out)):
        z.write(out + f, 'outputs/part3/' + f)
    z.write(path + 'q2_submission.csv', 'q2_submission.csv')
print(sorted(os.listdir(out)))

['accuracy_by_length.csv', 'ensemble_weights.csv', 'final_holdout_predictions.parquet', 'holdout_predictions.parquet', 'length_coverage.csv', 'model_comparison.csv', 'model_selection.csv', 'test_order_check.csv', 'test_prediction_shares.csv', 'test_row_blocks.csv', 'tfidf_model.npz', 'tfidf_valid_predictions.parquet', 'tfidf_vocabulary.json', 'train_mix_accuracy.csv', 'transformer_history.csv', 'valid_recall.csv']


## Summary

<div dir="rtl">

- روی validation متوازن: DeBERTa-v3-base ۰.۷۰۳۳، RoBERTa-base ۰.۶۹۲۰، DistilRoBERTa ۰.۶۷۸۹، MiniLM ۰.۶۷۲۳ و TF-IDF ۰.۶۰۳۲. بیشترین سود ترنسفورمرها در امتیازهای ۲ تا ۴ است.
- ترکیب وزنی احتمال‌ها (DeBERTa ۰.۵۵، RoBERTa ۰.۳، MiniLM ۰.۱، TF-IDF ۰.۰۵) درستی out-of-fold ۰.۷۰۵۵ دارد. فاصله‌اش با DeBERTa (۰.۲ امتیاز) در حد نویز است، پس طبق قاعده‌ی از پیش تعیین‌شده DeBERTa مدل نهایی شد.
- holdout، فقط یک بار: درستی (همان micro-F1) ۰.۶۹۴۹ با بازه‌ی ۹۵٪ از ۰.۶۸۶ تا ۰.۷۰۴، و macro-F1 ۰.۶۹۴۵. ۹۷.۸۵٪ پیش‌بینی‌ها حداکثر یک ستاره خطا دارند.
- فقط حدود ۳٪ نظرها از ۵۱۲ token بلندترند. سود DeBERTa نسبت به TF-IDF با بلندتر شدن نظر بیشتر می‌شود: ۶.۳ امتیاز در نظرهای کوتاه و ۱۴ امتیاز در نظرهای ۲۵۷ تا ۵۱۲ token.
- DistilRoBERTa، که بعد از انتخاب مدل نهایی برای مقایسه با کار یکی از اعضای گروه اضافه شد، روی holdout ۰.۶۸۲۵ است. نصف کردن لایه‌های RoBERTa حدود ۱ امتیاز هزینه داشت.
- فقط ۰.۳٪ پیش‌بینی‌ها سه ستاره یا بیشتر خطا دارند، و بیشترشان نظرهایی‌اند که متن و امتیازشان با هم نمی‌خوانند.
- همین مدل روی داده‌ای با ترکیب train ۰.۷۶ تا ۰.۷۹ درست می‌بود. عددهای روی این نوع داده با test متوازن قابل مقایسه نیستند.
- سهم امتیازهای پیش‌بینی‌شده روی test حدود ۲۰٪ برای هر امتیاز است، و سطرهای test به ترتیب امتیاز مرتب‌اند. با این فرض، درستی DeBERTa روی test ۰.۶۹۸۹ تخمین زده می‌شود که با holdout می‌خواند. ترتیب سطرها در هیچ پیش‌بینی‌ای استفاده نشد.
- q2_submission.csv: ۲۰٬۰۰۰ سطر با پیش‌بینی‌های DeBERTa.

</div>